# Role Label Standardization for Role Categorization

## Purpose of this Notebook

This notebook prepares the Djinni Job Descriptions dataset for the role categorization experiments.

The original dataset contains a `Position` column, which represents the raw job title for each job description. However, these job titles are not always consistent. For example, similar roles may appear with different names such as `Java Developer`, `Backend Engineer`, `Spring Developer`, or `Software Developer`.

To evaluate role categorization correctly, these raw job titles need to be converted into a fixed set of standardized role labels.

## What this Notebook Does

This notebook performs the following steps:

1. Loads the 100-sample Djinni dataset from Google Drive.
2. Uses the `Position` column as the raw ground-truth role label.
3. Defines a fixed role taxonomy, such as:
   - Backend Developer
   - Frontend Developer
   - Full Stack Developer
   - DevOps Engineer
   - Data / AI Role
   - QA Engineer
   - Mobile Developer
   - Product Manager
   - Project / Program Manager
   - Marketing / Sales
   - Business / Operations Role
   - Other
4. Applies rule-based keyword mapping to convert each raw `Position` value into a standardized `standard_role`.
5. Saves the role taxonomy separately as `role_taxonomy.csv` so that the label design is transparent and easy to inspect.
6. Saves the processed dataset with the new `standard_role` column for use in all role categorization experiments.

## Why this Step is Important

Raw job titles are often noisy and semantically overlapping. For example:

| Raw Position | Standardized Role |
|---|---|
| Java Developer | Backend Developer |
| Spring Boot Developer | Backend Developer |
| React Developer | Frontend Developer |
| Integration Specialist / DevOps | DevOps Engineer |
| QA Automation Engineer | QA Engineer |

Without this standardization step, evaluation would be unreliable because the model prediction and ground-truth label may use different wording even when they refer to the same role.

For example:

- Ground truth: `Java Developer`
- Model prediction: `Backend Developer`

These are semantically similar, but direct string comparison would mark them as different. Therefore, standardized labels are required for fair evaluation.

## Output Files Generated

This notebook generates the following files:

| File | Purpose |
|---|---|
| `role_taxonomy.csv` | Documents the fixed role categories used in the study |
| `djinni_with_reference_summaries.csv` | Processed dataset with standardized `true_role` column |
| `role_distribution.csv` | Summary of how many samples belong to each standardized role |

## How this Fits into the Research Pipeline

This notebook is part of the data preprocessing stage.

The processed dataset generated here will be reused in:

1. Role categorization without RAG
2. Role categorization with RAG
3. Prompt strategy comparisons
4. Accuracy, precision, recall, F1-score, and confusion matrix evaluation

In [1]:
# ================================================================
# 1. BASIC PYTHON LIBRARIES
# ================================================================
import re
import os
import shutil
import warnings
import importlib.metadata
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 60)
print("ROLE LABEL STANDARDIZATION NOTEBOOK")
print("=" * 60)
print("Purpose:")
print("1. Load Djinni 100-sample dataset from Google Drive")
print("2. Create fixed role taxonomy")
print("3. Standardize Position column into standard_role")
print("4. Save role_taxonomy.csv for examiner")
print("5. Save processed dataset with standardized roles")
print("=" * 60)

# ================================================================
# 2. VERSION CHECKS
# ================================================================

def get_version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except Exception:
        return "installed"

print("=" * 50)
print("LIBRARY VERSION CHECK")
print("=" * 50)
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)
print("=" * 50)

# ================================================================
# 3. MOUNT GOOGLE DRIVE
# ================================================================
from google.colab import drive

drive.mount("/content/drive")

print("\nGoogle Drive mounted successfully.")

# ================================================================
# 4. DEFINE PROJECT PATHS
# ================================================================
# Change this path only if your folder name is different.

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Masters/LJMU-masters-thesis")

DATA_RAW_DIR = PROJECT_DIR / "data" / "reference_summaries"
DATA_PROCESSED_DIR = PROJECT_DIR / "data" / "role_labels"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
LOGS_DIR = OUTPUTS_DIR / "logs"

DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = DATA_RAW_DIR / "djinni_with_reference_summaries.csv"

ROLE_TAXONOMY_FILE = DATA_PROCESSED_DIR / "role_taxonomy.csv"
OUTPUT_FILE = DATA_PROCESSED_DIR / "djinni_with_standardized_roles.csv"

print("=" * 60)
print("PROJECT PATHS")
print("=" * 60)
print("Project directory       :", PROJECT_DIR)
print("Raw data directory      :", DATA_RAW_DIR)
print("Processed data directory:", DATA_PROCESSED_DIR)
print("Input file              :", INPUT_FILE)
print("Role taxonomy file      :", ROLE_TAXONOMY_FILE)
print("Output file             :", OUTPUT_FILE)
print("=" * 60)

# ================================================================
# 5. LOAD DATASET
# ================================================================

print("Loading Djinni dataset...")

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found:\n{INPUT_FILE}\n\n"
        "Please upload/copy your CSV into data/raw/ folder."
    )

df = pd.read_csv(INPUT_FILE)

print("\nDataset loaded successfully.")
print("Total rows :", len(df))
print("Columns    :", list(df.columns))

print("\nSample records:")
display(df.head(3))

# ================================================================
# 6. VALIDATE REQUIRED COLUMN
# ================================================================

REQUIRED_COLUMN = "Position"

if REQUIRED_COLUMN not in df.columns:
    raise ValueError(
        f"Required column '{REQUIRED_COLUMN}' not found.\n"
        f"Available columns: {list(df.columns)}"
    )

print("=" * 60)
print("POSITION COLUMN VALIDATION")
print("=" * 60)
print("Required column found:", REQUIRED_COLUMN)
print("Null positions       :", df[REQUIRED_COLUMN].isna().sum())
print("Unique positions     :", df[REQUIRED_COLUMN].nunique())
print("=" * 60)

print("\nTop raw Position values:")
display(df[REQUIRED_COLUMN].value_counts().head(30).reset_index().rename(
    columns={"index": "Position", REQUIRED_COLUMN: "count"}
))

# ================================================================
# 7. CREATE FIXED ROLE TAXONOMY
# ================================================================
# This taxonomy is used as the fixed classification space.
# It should be reused in both:
# 1. Role categorization without RAG
# 2. Role categorization with RAG

role_taxonomy = [
    {
        "role_id": "ROLE_01",
        "standard_role": "Backend Developer",
        "description": "Server-side development roles involving backend logic, APIs, databases, Java, Python, Node.js, PHP, .NET, Golang, Spring, Django, Laravel, or microservices.",
        "example_keywords": "backend, java, spring, python developer, node.js, php, .net, golang, api, microservices, database"
    },
    {
        "role_id": "ROLE_02",
        "standard_role": "Frontend Developer",
        "description": "Client-side development roles involving web UI, JavaScript frameworks, HTML, CSS, React, Angular, Vue, or frontend engineering.",
        "example_keywords": "frontend, front-end, react, angular, vue, javascript, typescript, html, css, ui developer"
    },
    {
        "role_id": "ROLE_03",
        "standard_role": "Full Stack Developer",
        "description": "Roles requiring both frontend and backend development responsibilities.",
        "example_keywords": "full stack, full-stack, fullstack, mern, mean"
    },
    {
        "role_id": "ROLE_04",
        "standard_role": "DevOps Engineer",
        "description": "Infrastructure, CI/CD, cloud, deployment, automation, integration, Kubernetes, Docker, SRE, or system operations roles.",
        "example_keywords": "devops, cloud, aws, azure, gcp, kubernetes, docker, ci/cd, infrastructure, sre, integration"
    },
    {
        "role_id": "ROLE_05",
        "standard_role": "Data / AI Role",
        "description": "Data science, data engineering, machine learning, artificial intelligence, NLP, BI, or data analytics roles.",
        "example_keywords": "data scientist, data engineer, machine learning, ai, ml, nlp, data analyst, data analytics, business intelligence"
    },
    {
        "role_id": "ROLE_06",
        "standard_role": "QA Engineer",
        "description": "Software testing, quality assurance, automation testing, manual testing, SDET, and test engineering roles.",
        "example_keywords": "qa, quality assurance, tester, test engineer, automation testing, manual testing, sdet"
    },
    {
        "role_id": "ROLE_07",
        "standard_role": "Mobile Developer",
        "description": "Mobile application development roles involving Android, iOS, Flutter, React Native, Kotlin, or Swift.",
        "example_keywords": "android, ios, mobile, flutter, react native, kotlin, swift"
    },
    {
        "role_id": "ROLE_08",
        "standard_role": "Project / Program Manager",
        "description": "Project management, delivery management, scrum master, program management, and coordination roles.",
        "example_keywords": "project manager, program manager, delivery manager, scrum master, coordinator"
    },
    {
        "role_id": "ROLE_09",
        "standard_role": "Product Manager",
        "description": "Product ownership, product strategy, product management, and product leadership roles.",
        "example_keywords": "product manager, product owner, product lead"
    },
    {
        "role_id": "ROLE_10",
        "standard_role": "Marketing / Sales",
        "description": "Marketing, sales, business development, customer acquisition, lead generation, growth, PR, and account management roles.",
        "example_keywords": "marketing, sales, business development, growth, lead generation, account manager, pr manager"
    },
    {
        "role_id": "ROLE_11",
        "standard_role": "Business / Operations Role",
        "description": "Business, operations, HR, people operations, recruitment, finance, administration, customer support, and non-technical business roles.",
        "example_keywords": "hr, people partner, recruiter, operations, finance, customer support, business analyst, country manager"
    },
    {
        "role_id": "ROLE_12",
        "standard_role": "Design / Creative Role",
        "description": "Design, UX/UI, graphic design, product design, illustration, creative writing, content creation, storytelling, and visual communication roles.",
        "example_keywords": "designer, ux, ui, product designer, graphic designer, illustrator, creative designer, writer, content writer, storyteller"
    },
    {
        "role_id": "ROLE_13",
        "standard_role": "Other",
        "description": "Roles that do not clearly fit into the predefined taxonomy.",
        "example_keywords": "unclear, mixed, insufficient information"
    }
]

taxonomy_df = pd.DataFrame(role_taxonomy)

taxonomy_df.to_csv(ROLE_TAXONOMY_FILE, index=False)

print("=" * 60)
print("ROLE TAXONOMY CREATED")
print("=" * 60)
print("Number of role categories:", len(taxonomy_df))
print("Saved taxonomy file      :", ROLE_TAXONOMY_FILE)
print("=" * 60)

display(taxonomy_df)

# ================================================================
# 8. TEXT NORMALIZATION FUNCTION
# ================================================================

def normalize_text(text):
    """
    Normalizes raw position text for keyword-based role standardization.
    """
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r"[/|,+]", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\+\#\.\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ================================================================
# 9. ROLE STANDARDIZATION FUNCTION
# ================================================================
# Important:
# Order matters.
# More specific roles should be checked before generic roles.

def standardize_role(position):
    text = normalize_text(position)

    if text == "":
        return "Other"

    if any(keyword in text for keyword in [
        "full stack", "full-stack", "fullstack", "mern", "mean"
    ]):
        return "Full Stack Developer"

    if any(keyword in text for keyword in [
        "android", "ios", "mobile", "flutter", "react native", "kotlin", "swift"
    ]):
        return "Mobile Developer"

    if any(keyword in text for keyword in [
        "frontend", "front end", "front-end", "react", "angular", "vue",
        "javascript", "typescript", "html", "css", "ui developer"
    ]):
        return "Frontend Developer"

    if any(keyword in text for keyword in [
        "backend", "back end", "back-end",
        "java", "spring",
        "node", "node.js",
        "php", ".net", "dotnet",
        "golang", "go developer",
        "api developer", "server side", "server-side",
        "laravel", "django",
        "microservices",
        "database", "sql", "nosql",
        "ms sql dba", "sql dba", "database administrator", "dba"
    ]):
        return "Backend Developer"

    if any(keyword in text for keyword in [
        "devops", "dev ops", "cloud", "aws", "azure", "gcp", "kubernetes",
        "docker", "ci cd", "cicd", "infrastructure", "sre", "site reliability",
        "system administrator", "sysadmin", "integration specialist", "integration"
    ]):
        return "DevOps Engineer"

    if any(keyword in text for keyword in [
        "data scientist", "data engineer", "machine learning", "ml engineer",
        "ai engineer", "artificial intelligence", "nlp", "data analyst",
        "data analytics", "business intelligence", "bi developer", "big data"
    ]):
        return "Data / AI Role"

    if any(keyword in text for keyword in [
        "qa", "quality assurance", "tester", "testing", "test engineer",
        "automation engineer", "manual tester", "sdet"
    ]):
        return "QA Engineer"

    if any(keyword in text for keyword in [
        "product manager", "product owner", "product lead"
    ]):
        return "Product Manager"

    if any(keyword in text for keyword in [
        "project manager", "program manager", "delivery manager",
        "delivery management", "scrum master", "project coordinator",
        "program coordinator"
    ]):
        return "Project / Program Manager"

    if any(keyword in text for keyword in [
        "marketing", "sales", "business development", "growth", "lead generation",
        "account manager", "customer success", "seo", "smm", "content manager",
        "affiliate manager", "account affiliate manager", "pr manager"
    ]):
        return "Marketing / Sales"

    if any(keyword in text for keyword in [
        "hr", "human resources", "recruiter", "recruitment", "talent acquisition",
        "people partner", "people operations", "people manager",
        "operations", "finance", "accountant", "admin", "administrator",
        "customer support", "support specialist", "business analyst",
        "office manager", "country manager"
    ]):
        return "Business / Operations Role"

    if any(keyword in text for keyword in [
        "designer", "ux designer", "ui designer", "ui ux", "ux ui",
        "product designer", "graphic designer", "illustrator",
        "creative designer", "game designer", "writer", "content writer",
        "english writer", "storyteller", "copywriter", "creative"
    ]):
        return "Design / Creative Role"

    return "Other"



# ================================================================
# 10. APPLY ROLE STANDARDIZATION
# ================================================================

print("=" * 60)
print("APPLYING ROLE STANDARDIZATION")
print("=" * 60)

df["position_normalized"] = df["Position"].apply(normalize_text)
df["standard_role"] = df["Position"].apply(standardize_role)

print("Role standardization completed.")
print("New columns added:")
print("1. position_normalized")
print("2. standard_role")
print("=" * 60)

display(df[["Position", "position_normalized", "standard_role"]].head(20))


# ================================================================
# 11. ROLE DISTRIBUTION CHECK
# ================================================================

role_distribution = (
    df["standard_role"]
    .value_counts()
    .reset_index()
)

role_distribution.columns = ["standard_role", "count"]
role_distribution["percentage"] = round((role_distribution["count"] / len(df)) * 100, 2)

print("=" * 60)
print("STANDARDIZED ROLE DISTRIBUTION")
print("=" * 60)
display(role_distribution)

other_count = int(df[df["standard_role"] == "Other"].shape[0])
other_percentage = round((other_count / len(df)) * 100, 2)

print("=" * 60)
print("QUALITY CHECK")
print("=" * 60)
print("Total rows        :", len(df))
print("Other count       :", other_count)
print("Other percentage  :", other_percentage, "%")

if other_percentage > 20:
    print("\nWARNING:")
    print("The 'Other' category is greater than 20%.")
    print("You should inspect those rows and improve the keyword mapping.")
else:
    print("\nRole mapping looks acceptable.")
print("=" * 60)



# ================================================================
# 12. INSPECT ROWS MAPPED TO OTHER
# ================================================================

df_other = df[df["standard_role"] == "Other"][["Position", "position_normalized", "standard_role"]]

print("=" * 60)
print("ROWS MAPPED TO OTHER")
print("=" * 60)
print("Total Other rows:", len(df_other))

display(df_other.head(50))

# ================================================================
# 13. SAVE STANDARDIZED DATASET
# ================================================================

df.to_csv(OUTPUT_FILE, index=False)

print("=" * 60)
print("STANDARDIZED DATASET SAVED")
print("=" * 60)
print("Saved file:", OUTPUT_FILE)
print("Total rows:", len(df))
print("Columns   :", list(df.columns))
print("=" * 60)


# ================================================================
# 14. SAVE ROLE DISTRIBUTION REPORT
# ================================================================

ROLE_DISTRIBUTION_FILE = DATA_PROCESSED_DIR / "role_distribution.csv"

role_distribution.to_csv(ROLE_DISTRIBUTION_FILE, index=False)

print("=" * 60)
print("ROLE DISTRIBUTION REPORT SAVED")
print("=" * 60)
print("Saved file:", ROLE_DISTRIBUTION_FILE)
print("=" * 60)

# ================================================================
# 15. FINAL VALIDATION BEFORE USING IN EXPERIMENTS
# ================================================================

print("=" * 70)
print("FINAL VALIDATION SUMMARY")
print("=" * 70)

print("Input dataset:")
print(" ", INPUT_FILE)

print("\nGenerated files:")
print("1.", ROLE_TAXONOMY_FILE)
print("2.", OUTPUT_FILE)
print("3.", ROLE_DISTRIBUTION_FILE)

print("\nDataset shape:")
print(df.shape)

print("\nRequired experiment columns:")
required_columns_for_role_classification = [
    "Position",
    "Long Description",
    "Reference_Summary",
    "standard_role"
]

for col in required_columns_for_role_classification:
    if col in df.columns:
        print(f"FOUND     : {col}")
    else:
        print(f"NOT FOUND : {col}")

print("\nRole categories:")
for role in sorted(df["standard_role"].unique()):
    print("-", role)

print("=" * 70)
print("Role label standardization completed successfully.")
print("Use the standardized CSV for role categorization experiments.")
print("=" * 70)

ROLE LABEL STANDARDIZATION NOTEBOOK
Purpose:
1. Load Djinni 100-sample dataset from Google Drive
2. Create fixed role taxonomy
3. Standardize Position column into standard_role
4. Save role_taxonomy.csv for examiner
5. Save processed dataset with standardized roles
LIBRARY VERSION CHECK
pandas : 2.2.2
numpy  : 2.0.2
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Google Drive mounted successfully.
PROJECT PATHS
Project directory       : /content/drive/MyDrive/Colab Notebooks/Masters/LJMU-masters-thesis
Raw data directory      : /content/drive/MyDrive/Colab Notebooks/Masters/LJMU-masters-thesis/data/reference_summaries
Processed data directory: /content/drive/MyDrive/Colab Notebooks/Masters/LJMU-masters-thesis/data/role_labels
Input file              : /content/drive/MyDrive/Colab Notebooks/Masters/LJMU-masters-thesis/data/reference_summaries/djinni_with_reference_summaries.csv
Role taxonomy file      : /c

,Position,Long Description,Company Name,Exp Years,Primary Keyword,English Level,Published,Long Description_lang,id,index_level_0,Reference_Summary
0,Java Developer,The job holder will . Design develop and maint...,Quickstarter,3y,Java,upper,2021-12-01T00:00:00+02:00,en,67aefec1-1efc-5c5b-b2db-c0ef5b74f519,77261,Role Overview:\nThe Java Developer will design...
1,Integration specialist / DevOps,Project Description Join our Development Cente...,Luxoft,3y,DevOps,upper,2022-04-01T00:00:00+03:00,en,2ec599d6-1509-5c34-9aec-54ac2e04b2d5,72279,Role Overview:\nThe Integration Specialist / D...
2,Head of Marketing,TurnKey Labs is the most trusted offshore soft...,TurnKey Labs,5y,Marketing,upper,2021-10-01T00:00:00+03:00,en,edba8bfd-8a91-5b37-9f72-bde92397e804,69706,"Role Overview:\nReporting to the CEO, the Head..."


POSITION COLUMN VALIDATION
Required column found: Position
Null positions       : 0
Unique positions     : 98

Top raw Position values:


,count,count
0,People Partner,2
1,Project Manager,2
2,Integration specialist / DevOps,1
3,Java Developer,1
4,Sales manager,1
5,Head of Marketing,1
6,PHP Developer,1
7,"MERN (Node, React) JS Full-stack developer",1
8,Golang Engineer,1
9,Creative Designer,1


ROLE TAXONOMY CREATED
Number of role categories: 13
Saved taxonomy file      : /content/drive/MyDrive/Colab Notebooks/Masters/LJMU-masters-thesis/data/role_labels/role_taxonomy.csv


...

STANDARDIZED ROLE DISTRIBUTION


,standard_role,count,percentage
0,Backend Developer,19,19.0
1,Other,13,13.0
2,Marketing / Sales,10,10.0
3,Business / Operations Role,9,9.0
4,Design / Creative Role,9,9.0
5,DevOps Engineer,8,8.0
6,Frontend Developer,8,8.0
7,QA Engineer,7,7.0
8,Full Stack Developer,6,6.0
9,Mobile Developer,4,4.0


QUALITY CHECK
Total rows        : 100
Other count       : 13
Other percentage  : 13.0 %

Role mapping looks acceptable.
ROWS MAPPED TO OTHER
Total Other rows: 13


,Position,position_normalized,standard_role
15,Analyst Relations Manager,analyst relations manager,Other
17,WFM Real Time Analyst,wfm real time analyst,Other
21,Performance Engineer (Python),performance engineer python,Other
30,Engineering Manager (Digital Lead),engineering manager digital lead,Other
38,Intern С# Developer,intern # developer,Other
39,Embedded Software Engineer,embedded software engineer,Other
51,Unreal Engine 3D developer,unreal engine 3d developer,Other
69,Clinical Design Engineer,clinical design engineer,Other
78,Quality Analyst,quality analyst,Other
79,Sl3 support/Operation Engineer,sl3 support operation engineer,Other


STANDARDIZED DATASET SAVED
Saved file: /content/drive/MyDrive/Colab Notebooks/Masters/LJMU-masters-thesis/data/role_labels/djinni_with_standardized_roles.csv
Total rows: 100
Columns   : ['Position', 'Long Description', 'Company Name', 'Exp Years', 'Primary Keyword', 'English Level', 'Published', 'Long Description_lang', 'id', 'index_level_0', 'Reference_Summary', 'position_normalized', 'standard_role']
ROLE DISTRIBUTION REPORT SAVED
Saved file: /content/drive/MyDrive/Colab Notebooks/Masters/LJMU-masters-thesis/data/role_labels/role_distribution.csv
FINAL VALIDATION SUMMARY
Input dataset:
  /content/drive/MyDrive/Colab Notebooks/Masters/LJMU-masters-thesis/data/reference_summaries/djinni_with_reference_summaries.csv

Generated files:
1. /content/drive/MyDrive/Colab Notebooks/Masters/LJMU-masters-thesis/data/role_labels/role_taxonomy.csv
2. /content/drive/MyDrive/Colab Notebooks/Masters/LJMU-masters-thesis/data/role_labels/djinni_with_standardized_roles.csv
3. /content/drive/MyDrive/Cola